# B2-024 — Practice p16

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** core · **Time budget:** 65 minutes  
**Concepts:** gpu-colab-l4-workflow

## Task

Train a small CNN with the p06/p07 helpers, checkpoint it mid-run, resume it in a fresh model, and certify both learning and an exact resume on the CPU.

**Data.** `shapes_supervised` has 900 synthetic $8\times8$ grey images in $[0,1]$, float32 `(N, 1, 8, 8)`: class 0 is a square frame, class 1 a plus, class 2 a diagonal cross, each with random size, stroke width, contrast, a sub-pixel position jitter, and heavy pixel noise. Use `x, y = capstone_data.train_rows("shapes_supervised")` (600 rows, int64 labels) for training and `capstone_data.val_rows("shapes_supervised")` (150 rows) for validation.

**Helpers.** Re-implement in this notebook `get_device`, `move_batch`, `train_step` (p06 contract) and `save_checkpoint`, `load_checkpoint` (p07 contract).

**Model.** `class ShapeCNN(nn.Module)` with constructor `ShapeCNN(channels=(8, 16))` creating, in this order, `self.conv1 = nn.Conv2d(1, channels[0], 3)`, `self.conv2 = nn.Conv2d(channels[0], channels[1], 3)`, `self.head = nn.Linear(channels[1] * 4 * 4, 3)`; `forward(x)` returns `self.head(F.relu(self.conv2(F.relu(self.conv1(x)))).flatten(1))`.
A *fresh run* is `torch.manual_seed(SEED)`, then `model = ShapeCNN(config["channels"]).to(device)`, then `optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`.

**One code path.** Every device, precision, size, and data knob is read from one config dict with the keys of the supplied `CPU_CONFIG`; the accelerator extension changes only the dictionary.
Write `train(config, model, optimizer, start_step, checkpoint_path)` with `device = torch.device(config["device"])`. Its sampling pool is `(x, y)` when `config["simulated_rows"] == 0`, and otherwise `(x, y)` concatenated with `capstone_data.simulate("shapes_supervised", config["simulated_rows"], seed=1)`.
For `step = start_step + 1, ..., config["steps"]` it draws `idx = torch.randint(0, n_pool, (config["batch_size"],))` from the global generator (`n_pool = 600` on the CPU), calls `train_step(model, (pool_x[idx], pool_y[idx]), optimizer, device, config["amp_dtype"])`, appends the returned loss, and calls `save_checkpoint(checkpoint_path, model, optimizer, step)` when `step == config["checkpoint_at"]`; it returns the list of losses. The CPU path uses exactly the supplied `CPU_CONFIG`.

**Seam (leakage check).** Before any training, register a forward pre-hook on each model that, while a flag `fitting` is true, adds `capstone_data.split_of("shapes_supervised", row)` for every row of the input batch to a set. Set `fitting = True` only around the calls to `train`.

**Runs.**

* *Uninterrupted:* fresh run; `losses = train(CPU_CONFIG, model, optimizer, 0, path)` (300 losses; it saves the checkpoint at step 150).
* *Resumed:* fresh run (a new model and optimizer); `torch.manual_seed(SEED + 99)` (a new session's generator is elsewhere); `k = load_checkpoint(path, model2, optimizer2)`; `resumed = train(CPU_CONFIG, model2, optimizer2, k, other_path)`.

**Evaluation protocol.** Fit only on rows from `capstone_data.train_rows("shapes_supervised")` (and any other rows this statement names), make every choice on `capstone_data.val_rows("shapes_supervised")`, and reach the locked test rows only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="shapes_supervised")`, called exactly once, after all fitting and selection. Assert `capstone_data.test_call_count("shapes_supervised") == 0` immediately before that call and `== 1` after it.
Score the **resumed** model; `predict_fn` returns `model2(images).argmax(dim=1)` (int64) under `torch.no_grad()`.

**Certify.** `k == 150`; `resumed == losses[150:]` exactly (150 Python floats) and every tensor of the two final `state_dict`s is `torch.equal`; the mean of the last 20 losses is `<= 0.25 *` the mean of the first 20; validation accuracy of the resumed model `>= 0.75`; the seam set is exactly `{"train"}`; the test-call protocol; the reported test accuracy `>= 0.75`, printed with its interval.
The thresholds were set from the frozen-seed reference run (loss ratio about `0.03`, validation accuracy `0.88`) with wide margins.

**Required answer-check assertions.** Every certificate item, plus `CPU_CONFIG["amp_dtype"] is None`.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import os
import tempfile

CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "batch_size": 64, "steps": 300, "checkpoint_at": 150,
              "lr": 0.01, "channels": (8, 16), "simulated_rows": 0}

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

The same `train(config, ...)` and fresh-run code run on a Colab L4 with `L4_CONFIG` in place of `CPU_CONFIG`; it has exactly the same keys, so only the dictionary changes. With `simulated_rows = 60000`, `train` samples `idx` over `x, y` plus 60,000 rows simulated from the same generative process (`capstone_data.simulate("shapes_supervised", 60000, seed=1)`). Simulated rows hash to `None`, never to `"val"` or `"test"`, so the seam check becomes "no `val` or `test` rows".

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "batch_size": 1024, "steps": 6000,
             "checkpoint_at": 3000, "lr": 0.003, "channels": (32, 64), "simulated_rows": 60000}
```

**What to observe.** Validation accuracy versus the number of simulated rows; steps per second with `amp_dtype=None` versus `torch.bfloat16`; peak memory against Session 1's estimate for `channels=(32, 64)`; and whether the resumed losses match the uninterrupted ones exactly on the GPU (they may not, without deterministic algorithms).

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.